# Iris Flower Classification using Machine Learning

## Objective

**What is the problem?**
The problem is to classify Iris flowers into three species (Setosa, Versicolor, Virginica) based on their sepal and petal dimensions.

**Why is it useful?**
It demonstrates basic classification techniques and is a standard benchmark for comparing ML algorithms.

**Expected outcome.**
A trained machine learning model capable of accurately classifying new Iris flowers based on feature measurements.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import joblib
import warnings
warnings.filterwarnings('ignore')

## Dataset Loading
We will load the Iris dataset from Scikit-Learn.

In [ ]:
iris = load_iris()
df = pd.DataFrame(data=iris.data, columns=iris.feature_names)
df['species'] = iris.target
df['species'] = df['species'].map({0: 'setosa', 1: 'versicolor', 2: 'virginica'})

In [ ]:
display(df.head())
display(df.tail())
display(df.sample(5))
print('Shape:', df.shape)
print('Columns:', df.columns)
df.info()
display(df.describe())
print(df.dtypes)

## Data Cleaning
Check for null values, duplicates, and correct datatypes.

In [ ]:
print('Null values:\n', df.isnull().sum())
print('Duplicates:', df.duplicated().sum())
# Dropping duplicates if any
df.drop_duplicates(inplace=True)
print('Shape after dropping duplicates:', df.shape)

We observed 1 duplicate which has been dropped. No null values are present. Datatypes are correct.

## Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(10, 8))
sns.pairplot(df, hue='species', palette='Set2')
plt.suptitle('Pairplot of Iris Features', y=1.02)
plt.savefig('../images/pairplot.png')
plt.show()

### Observations
- Setosa is clearly separable from the other two species in almost all feature combinations.
- Virginica and Versicolor overlap slightly, especially in sepal measurements.
- Petal Length and Petal Width are highly discriminative.

In [ ]:
plt.figure(figsize=(10, 8))
sns.boxplot(data=df, orient='h', palette='Set2')
plt.title('Boxplots of Iris Features')
plt.savefig('../images/boxplots.png')
plt.show()

### Observations
- Sepal width has a few outliers, but they are natural variations. The scales of features are relatively similar.

In [ ]:
df.hist(figsize=(10,8), color='teal', edgecolor='black')
plt.suptitle('Histograms of Features')
plt.savefig('../images/histograms.png')
plt.show()

### Observations
- Sepal length and width appear to follow a somewhat normal distribution.
- Petal measurements show bi-modal distributions, largely because Setosa is distinctly separated from the rest.

In [ ]:
plt.figure(figsize=(8, 6))
corr = df.drop('species', axis=1).corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Heatmap')
plt.savefig('../images/correlation_heatmap.png')
plt.show()

### Observations
- Petal length and petal width are highly correlated (0.96).
- Sepal length and petal length are also highly correlated (0.87).
- Sepal width is inversely correlated with petal length and width.

## Feature Discussion
- **Sepal Length & Width**: Show more overlap between Versicolor and Virginica.
- **Petal Length & Width**: Provide the best separation for all three species. These are the most critical features for the models.

## Feature Engineering
We don't need extensive feature engineering here because the measurements are already robust and clean. We'll simply use the 4 numeric features to predict the species.

In [ ]:
X = df.drop('species', axis=1)
y = df['species']

## Train Test Split
We split the data using an 80/20 ratio and `random_state=42` for reproducibility.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('Training data shape:', X_train.shape)
print('Test data shape:', X_test.shape)

## Model Training
We will train Logistic Regression, K-Nearest Neighbors, Decision Tree, and Random Forest models. These cover linear, distance-based, and tree-based approaches.

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=200),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42)
}

results = {}

## Model Evaluation

In [ ]:
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    results[name] = acc
    print(f'--- {name} ---')
    print('Accuracy:', acc)
    print(classification_report(y_test, y_pred))
    
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, cmap='Blues', fmt='g')
    plt.title(f'Confusion Matrix - {name}')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.savefig(f'../images/cm_{name.replace(" ", "_")}.png')
    plt.show()

### Metric Explanations
- **Accuracy**: Overall proportion of correct predictions.
- **Precision**: Proportion of positive identifications that were actually correct.
- **Recall**: Proportion of actual positives that were identified correctly.
- **F1 Score**: Harmonic mean of Precision and Recall, providing a balance between them.

## Best Model Selection
Comparing all models based on accuracy and robust performance.

In [ ]:
best_model_name = max(results, key=results.get)
print(f'The best model is {best_model_name} with an accuracy of {results[best_model_name]}')

### Why it won
- The models generally perform similarly (often 100% on this small dataset with a good split).
- Random Forest or Logistic Regression are excellent choices. Random Forest provides feature importance and handles complex boundaries, making it highly advantageous, while its limitation is less interpretability compared to a single Decision Tree.

## Save Best Model

In [ ]:
best_model = models[best_model_name]
joblib.dump(best_model, '../models/best_model.pkl')
print('Best model saved to ../models/best_model.pkl')

## Final Conclusion
- **Insights**: Petal dimensions are the best indicators for Iris species classification. Setosa is entirely distinct, while Versicolor and Virginica have minor boundary overlaps.
- **Model Performance**: Extremely high accuracy achieved across all algorithms, highlighting the simplicity and quality of this dataset.
- **Future Improvements**: In a production environment, exploring hyperparameter tuning (e.g., GridSearchCV) or cross-validation would ensure robustness against varied data splits.